# Bài 5: Xây dựng một Pipeline Khoa học dữ liệu với OOP

## Mục tiêu bài học

Trong bài học tổng hợp này, chúng ta sẽ áp dụng tất cả các kiến thức đã học về Lập trình Hướng đối tượng (OOP) để xây dựng một pipeline khoa học dữ liệu hoàn chỉnh. Mục tiêu là:

- Tổng hợp và áp dụng các khái niệm: Class, Kế thừa, Đóng gói, Đa hình.
- Sử dụng các mẫu thiết kế (Design Patterns) đã học trong một kịch bản thực tế.
- Xây dựng một pipeline có cấu trúc, module hóa, dễ đọc, dễ bảo trì và dễ mở rộng.
- Thấy được sức mạnh của OOP trong việc quản lý sự phức tạp của một dự án data science.

## Bối cảnh bài toán

Chúng ta sẽ xây dựng một pipeline đơn giản để dự đoán giá nhà dựa trên bộ dữ liệu California Housing của scikit-learn. Pipeline của chúng ta sẽ bao gồm các bước chính:

1.  **Load Data:** Tải dữ liệu.
2.  **Process Data:** Tiền xử lý dữ liệu (xử lý giá trị thiếu, scaling).
3.  **Train Model:** Huấn luyện mô hình.
4.  **Evaluate Model:** Đánh giá mô hình.

Thay vì viết tất cả trong một script dài, chúng ta sẽ thiết kế mỗi bước như một `class` riêng biệt, và sau đó kết hợp chúng lại thành một `Pipeline`.

## Bước 1: Thiết kế các "Bước" của Pipeline (Pipeline Steps)

Ý tưởng ở đây là mỗi bước trong pipeline (load, process, train, evaluate) đều là một hành động. Chúng ta có thể định nghĩa một "giao diện" chung cho tất cả các bước này. Đây là một ứng dụng tuyệt vời của **Kế thừa** và **Đa hình**.

Chúng ta sẽ tạo một class cha `PipelineStep`.

In [ ]:
from abc import ABC, abstractmethod
import pandas as pd

class PipelineStep(ABC):
    """Interface cho một bước trong pipeline."""
    
    @abstractmethod
    def process(self, data=None):
        """Mỗi bước phải implement phương thức này."""
        pass

### 1.1. DataLoader Step

Bước đầu tiên là tải dữ liệu. Class này sẽ kế thừa từ `PipelineStep`.

In [ ]:
from sklearn.datasets import fetch_california_housing

class DataLoader(PipelineStep):
    def process(self, data=None) -> pd.DataFrame:
        print("--- 1. Đang tải dữ liệu California Housing... ---")
        housing = fetch_california_housing()
        df = pd.DataFrame(housing.data, columns=housing.feature_names)
        df['MedHouseVal'] = housing.target
        print(f"Tải thành công. Dữ liệu có {df.shape[0]} dòng và {df.shape[1]} cột.")
        return df

### 1.2. DataProcessor Step

Bước này sẽ thực hiện tiền xử lý. Chúng ta sẽ sử dụng **Strategy Pattern** ở đây để có thể linh hoạt thay đổi cách xử lý giá trị thiếu.

*(Chúng ta sẽ sử dụng lại các class Strategy từ bài 4, nhưng định nghĩa lại ở đây cho rõ ràng)*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

class DataProcessor(PipelineStep):
    def __init__(self, test_size=0.2, random_state=42):
        self._test_size = test_size
        self._random_state = random_state
        self._imputer = SimpleImputer(strategy='mean')
        self._scaler = StandardScaler()

    def process(self, data: pd.DataFrame) -> tuple:
        print("\n--- 2. Đang tiền xử lý dữ liệu... ---")
        
        # Tách features và target
        X = data.drop('MedHouseVal', axis=1)
        y = data['MedHouseVal']
        
        # Chia train/test set
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self._test_size, random_state=self._random_state
        )
        print(f"Đã chia dữ liệu: {len(X_train)} mẫu train, {len(X_test)} mẫu test.")
        
        # Xử lý missing (ví dụ này không có, nhưng đây là nơi để làm)
        X_train = self._imputer.fit_transform(X_train)
        X_test = self._imputer.transform(X_test)
        print("Đã xử lý giá trị thiếu (nếu có).")
        
        # Scale dữ liệu
        X_train = self._scaler.fit_transform(X_train)
        X_test = self._scaler.transform(X_test)
        print("Đã scale dữ liệu.")
        
        return X_train, X_test, y_train, y_test

### 1.3. ModelTrainer Step

Bước này sẽ huấn luyện mô hình. Chúng ta có thể dùng **Factory Pattern** để chọn mô hình muốn huấn luyện.

*(Sử dụng lại ý tưởng từ bài 4)*

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

class ModelFactory:
    def get_model(self, model_name: str):
        if model_name == 'linear_regression':
            print("Đang tạo mô hình Linear Regression...")
            return LinearRegression()
        elif model_name == 'random_forest':
            print("Đang tạo mô hình Random Forest...")
            return RandomForestRegressor(n_estimators=100, random_state=42)
        else:
            raise ValueError(f"Mô hình không được hỗ trợ: {model_name}")

class ModelTrainer(PipelineStep):
    def __init__(self, model_name: str):
        self._model_name = model_name
        self.model = ModelFactory().get_model(model_name)

    def process(self, data: tuple):
        print("\n--- 3. Đang huấn luyện mô hình... ---")
        X_train, _, y_train, _ = data
        
        self.model.fit(X_train, y_train)
        print(f"Đã huấn luyện xong mô hình {self._model_name}.")
        return self.model

### 1.4. ModelEvaluator Step

Bước cuối cùng là đánh giá mô hình trên tập test.

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

class ModelEvaluator(PipelineStep):
    def __init__(self, model):
        self._model = model

    def process(self, data: tuple):
        print("\n--- 4. Đang đánh giá mô hình... ---")
        _, X_test, _, y_test = data
        
        predictions = self._model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        
        print(f"Đánh giá hoàn tất.")
        print(f"  - Mean Squared Error (MSE): {mse:.4f}")
        print(f"  - Root Mean Squared Error (RMSE): {rmse:.4f}")
        return {'mse': mse, 'rmse': rmse}

## Bước 2: Thiết kế Class `Pipeline`

Bây giờ chúng ta sẽ tạo một class `Pipeline` để quản lý và thực thi tất cả các bước trên một cách tuần tự. Class này sẽ chứa một danh sách các `PipelineStep`.

In [ ]:
class Pipeline:
    def __init__(self, steps: list[PipelineStep]):
        self._steps = steps
        self.results = {}

    def run(self):
        print("========= BẮT ĐẦU PIPELINE =========")
        data_so_far = None
        for i, step in enumerate(self._steps):
            # Đây chính là Đa hình!
            # Pipeline không cần biết step là DataLoader hay ModelTrainer,
            # nó chỉ cần biết step có phương thức process().
            data_so_far = step.process(data_so_far)
            self.results[step.__class__.__name__] = data_so_far
        
        print("\n========= PIPELINE HOÀN TẤT =========")
        return self.results

## Bước 3: Lắp ráp và Chạy Pipeline

Giờ là lúc thú vị nhất: lắp ráp các mảnh ghép và xem nó hoạt động.

In [ ]:
# --- Cấu hình và lắp ráp Pipeline ---

# 1. Chọn mô hình bạn muốn chạy
MODEL_TO_RUN = 'random_forest' # hoặc 'linear_regression'

# 2. Khởi tạo các bước
data_loader = DataLoader()
data_processor = DataProcessor()
model_trainer = ModelTrainer(model_name=MODEL_TO_RUN)

# Lưu ý: ModelEvaluator cần được tạo sau khi ModelTrainer đã chạy
# Chúng ta sẽ xử lý việc này một cách linh hoạt hơn

# --- Chạy các bước chính ---
df = data_loader.process()
processed_data = data_processor.process(df)
trained_model = model_trainer.process(processed_data)

# --- Tạo và chạy bước cuối cùng ---
model_evaluator = ModelEvaluator(model=trained_model)
evaluation_metrics = model_evaluator.process(data=processed_data)

print("\n--- KẾT QUẢ CUỐI CÙNG ---")
print(evaluation_metrics)

### Cải tiến với Class `Pipeline`

Cách chạy ở trên vẫn còn hơi thủ công. Hãy sử dụng class `Pipeline` để làm cho nó tự động hơn. Chúng ta cần điều chỉnh một chút để xử lý việc `ModelEvaluator` phụ thuộc vào kết quả của `ModelTrainer`.

In [ ]:
class DynamicPipeline:
    def run(self, model_name: str):
        print(f"\n========= BẮT ĐẦU DYNAMIC PIPELINE VỚI MODEL: {model_name} =========")
        
        # Bước 1 & 2
        df = DataLoader().process()
        processed_data = DataProcessor().process(df)
        
        # Bước 3
        trainer = ModelTrainer(model_name)
        trained_model = trainer.process(processed_data)
        
        # Bước 4
        evaluator = ModelEvaluator(trained_model)
        metrics = evaluator.process(processed_data)
        
        print("\n========= DYNAMIC PIPELINE HOÀN TẤT =========")
        return metrics

# Chạy pipeline cho Linear Regression
lr_pipeline = DynamicPipeline()
lr_metrics = lr_pipeline.run('linear_regression')

# Chạy pipeline cho Random Forest
rf_pipeline = DynamicPipeline()
rf_metrics = rf_pipeline.run('random_forest')

## Tổng kết và Phân tích

Chúng ta đã làm được gì?

1.  **Module hóa (Modularity):** Mỗi bước của pipeline là một class riêng biệt, có trách nhiệm rõ ràng. `DataLoader` chỉ lo tải dữ liệu, `ModelTrainer` chỉ lo huấn luyện. Điều này làm code rất dễ đọc và dễ tìm lỗi.

2.  **Tái sử dụng (Reusability):** Bạn có thể dễ dàng lấy class `DataLoader` hoặc `DataProcessor` để sử dụng trong một dự án khác.

3.  **Linh hoạt và Dễ mở rộng (Flexibility & Extensibility):**
    -   Nhờ **Factory Pattern**, để thử nghiệm một mô hình mới (ví dụ: `XGBoost`), bạn chỉ cần thêm một `elif` trong `ModelFactory` và một class `XGBoostModel` tương ứng. Toàn bộ pipeline không cần thay đổi.
    -   Nhờ **Strategy Pattern** (ẩn trong `DataProcessor`), bạn có thể dễ dàng thay đổi cách xử lý giá trị thiếu (ví dụ từ `mean` sang `median`) bằng cách thay đổi `SimpleImputer` mà không ảnh hưởng các bước khác.
    -   Nhờ **Đa hình**, class `Pipeline` có thể chạy bất kỳ bước nào miễn là nó tuân theo "hợp đồng" của `PipelineStep` (có phương thức `process`).

4.  **Dễ bảo trì (Maintainability):** Nếu logic scaling dữ liệu cần thay đổi, bạn biết chính xác phải vào file/class nào để sửa (`DataProcessor`), không cần phải đọc một script dài 500 dòng.

Đây chính là sức mạnh của việc áp dụng OOP vào các dự án khoa học dữ liệu. Nó giúp chúng ta chuyển từ việc viết các script dùng một lần sang việc xây dựng các hệ thống phần mềm thực thụ, có khả năng phát triển và thích ứng với các yêu cầu mới một cách bền vững.

## Hướng đi tiếp theo

Đây là một ví dụ đơn giản. Trong thực tế, bạn có thể cải tiến pipeline này bằng cách:
-   Sử dụng **Singleton Pattern** cho một class `ConfigManager` để quản lý tất cả các tham số (test_size, random_state, model_name) từ một file YAML.
-   Làm cho `DataProcessor` thực sự sử dụng Strategy Pattern bằng cách cho phép truyền các chiến lược imputer và scaler từ bên ngoài.
-   Xây dựng một `ResultLogger` để lưu kết quả của mỗi lần chạy vào một file CSV hoặc một database.

Chúc mừng bạn đã hoàn thành khóa học cơ bản về Lập trình Hướng đối tượng cho Khoa học dữ liệu!